# 04 · Resultados históricos congelados

Lê as tabelas **já geradas** pela avaliação congelada (`reports/atribuicao_final/analise/`). Nada é reexecutado aqui.

* **A**: mais próximo livre, despacho imediato · **B**: mais próximo livre, lotes de 60 s · **C**: Húngaro, lotes de 60 s.
* Contraste principal **C × A**; complementar **C × B** (efeito da atribuição com o lote fixo).
* Unidade de análise: o **dia**. Variação de km vazio: negativo = redução.
* A espera é calculada **só entre os pedidos atendidos**; políticas diferentes atendem conjuntos diferentes de pedidos.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "execute o notebook dentro do repositório"
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
SEED = 20261108
np.random.seed(SEED)

In [2]:
path = ROOT / "reports/atribuicao_final/analise/resumo.json"
if not path.exists():
    raise AssertionError("resultados congelados ausentes neste clone: gere-os com `tcc-frota analyze-historical` ou peça ao proprietário")
R = json.loads(path.read_text(encoding="utf-8"))
KEY = "variacao_km_vazio_pct (negativo = reducao)"


def contrast(period, name):
    return next(c for c in R[period]["contrastes"] if c["contraste"] == name)

In [3]:
rows = []
for period, label in (("replicacao", "Setembro 16–30/09 (15 dias)"), ("final", "Outubro 01–25/10 (25 dias)")):
    for name in ("C × A", "C × B", "B × A"):
        c = contrast(period, name)
        rows.append(
            {
                "período": label,
                "contraste": name,
                "Δ km vazio (%)": round(c[KEY], 2),
                "IC95% Δ média diária (km)": tuple(round(x, 1) for x in c["ic95_diferenca_media_diaria_km"]),
                "dias melhor/pior": f"{c['dias_com_melhora']}/{c['dias_com_piora']}",
                "Δ atendidos": c["diferenca_atendidos"],
                "Δ atendimento (p.p. do universo)": round(c["diferenca_atendimento_pontos_percentuais"], 2),
                "E1": c["E1_sem_violacoes"],
                "E2": c["E2_atendimento_preservado"],
                "E3": c["E3_p90_nao_piorou"],
            }
        )
table = pd.DataFrame(rows)
table

                       período contraste  Δ km vazio (%) IC95% Δ média diária (km) dias melhor/pior  Δ atendidos  Δ atendimento (p.p. do universo)    E1  \
0  Setembro 16–30/09 (15 dias)     C × A           -5.96          (-133.0, -102.9)             14/1           16                              0.09  True   
1  Setembro 16–30/09 (15 dias)     C × B           -2.74            (-61.2, -43.1)             14/1          191                              1.13  True   
2  Setembro 16–30/09 (15 dias)     B × A           -3.32            (-76.1, -54.8)             14/1         -175                             -1.03  True   
3   Outubro 01–25/10 (25 dias)     C × A           -5.04            (-97.3, -62.8)             25/0          -85                             -0.36  True   
4   Outubro 01–25/10 (25 dias)     C × B           -1.40            (-31.1, -11.5)             16/9          149                              0.63  True   
5   Outubro 01–25/10 (25 dias)     B × A           -3.69        

## O que aconteceu em outubro (C × A)

O km vazio caiu em **todos** os 25 dias, mas C atendeu **menos pedidos** que A. O critério **E2** (atendidos(C) ≥ atendidos(A)) **não foi cumprido**. Por isso o resultado de outubro **não** é uma redução com serviço preservado no sentido do protocolo, mesmo com intervalo de confiança do atendimento que inclui zero: *intervalo que inclui zero não é equivalência*.

In [4]:
c = contrast("final", "C × A")
ab = {r["politica"][0]: r for r in R["final"]["absolutos"]}
A, C = ab["A"], ab["C"]
assert (A["atendidos"], C["atendidos"], A["pedidos_elegiveis"]) == (14551, 14466, 23523)
assert c["diferenca_atendidos"] == -85 and c["E2_atendimento_preservado"] is False
print("atendidos A:", A["atendidos"], "| C:", C["atendidos"], "| universo elegível:", A["pedidos_elegiveis"])
print(f"Δ = {c['diferenca_atendidos']:d} pedidos")
print(f"−0,36 p.p.  = 85 / 23.523 (taxa sobre o UNIVERSO elegível): {100 * -85 / 23523:.2f} p.p.")
print(f"≈ −0,58%   = 85 / 14.551 (relativo aos ATENDIDOS de A):      {100 * -85 / 14551:.2f}%")
assert round(100 * -85 / 23523, 2) == -0.36 and round(100 * -85 / 14551, 2) == -0.58
print(f"Δ km vazio (%): {c[KEY]:.2f}  | atendimento de C no universo: {C['atendimento_pct']:.2f}%")

atendidos A: 14551 | C: 14466 | universo elegível: 23523
Δ = -85 pedidos
−0,36 p.p.  = 85 / 23.523 (taxa sobre o UNIVERSO elegível): -0.36 p.p.
≈ −0,58%   = 85 / 14.551 (relativo aos ATENDIDOS de A):      -0.58%
Δ km vazio (%): -5.04  | atendimento de C no universo: 61.50%


Em setembro, C × A atendeu +16 pedidos (E2 cumprido) e C × B atendeu +191; em outubro, C × B atendeu +149.

In [5]:
assert contrast("replicacao", "C × A")["diferenca_atendidos"] == 16 and contrast("replicacao", "C × B")["diferenca_atendidos"] == 191
assert contrast("final", "C × B")["diferenca_atendidos"] == 149
assert round(contrast("replicacao", "C × A")[KEY], 2) == -5.96 and round(contrast("final", "C × A")[KEY], 2) == -5.04

## Espera (somente atendidos) e custo computacional
Denominador = pedidos **atendidos** de cada política, que diferem entre políticas; não é uma melhoria garantida (viés de seleção possível).

In [6]:
w = pd.DataFrame(R["final"]["absolutos"])[["politica", "atendidos", "espera_media_s", "espera_p90_s", "tempo_computacional_por_dia_s"]]
w["politica"] = w.politica.str[0]
w.round(2)

  politica  atendidos  espera_media_s  espera_p90_s  tempo_computacional_por_dia_s
0        A      14551          620.98       1189.68                           0.05
1        B      14317          633.38       1201.45                           0.08
2        C      14466          615.81       1181.70                           0.10

`tempo_computacional_por_dia_s` mede o **relógio de parede da chamada `simulate` de um dia inteiro** daquela política (elegibilidade, distâncias, laço de lotes e o Húngaro), **sem** a construção do mundo e sem leitura de dados. Não é o tempo isolado do solver. Ambiente do registro: Python 3.12, execução local sequencial.

In [7]:
from IPython.display import display

from tcc_frota.visualization import figures

out = ROOT / "reports/figures"
figures.plot_km_and_attendance(R, out)
figures.plot_confidence_intervals(R, out)
from IPython.display import Image

display(Image(filename=str(out / "km_atendimento.png")))
display(Image(filename=str(out / "ic_diferenca_diaria.png")))

<IPython.core.display.Image object>

<IPython.core.display.Image object>